# Notebook 01c — NB1 Resume: Section E + F + G

**Purpose:** Resumes NB1 from where it crashed (Section E — `NameError: RATIOS`).
Assumes NB1 already ran and saved:
- `pre_train/theta_o_seed{s}.pt` for all seeds
- `splits/forget_indices_class{c}.json` + `retain_indices_class{c}.json`

This notebook:
1. Loads existing theta_o checkpoints (no retraining)
2. Runs Section E — Output / Linear Probe / NCC evaluation
3. Runs Section F — writes `cmf_benchmark_config.json` (required by NB4a)
4. Runs Section G — summary print

**Attach the NB1 output dataset as input so checkpoints are accessible.**

In [ ]:
import subprocess, sys
def sh(cmd, verbose=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if verbose and r.stdout: print(r.stdout[-4000:])
    if r.returncode != 0 and r.stderr: print('STDERR:', r.stderr[-2000:])
    return r.returncode
sh('pip install -q timm einops scikit-learn matplotlib seaborn pytorch-lightning torchmetrics')

In [ ]:
import os, sys, json, random, copy, math, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
print('PyTorch:', torch.__version__, '  CUDA:', torch.cuda.is_available())

In [ ]:
REPO_DIR = '/kaggle/working/CMF_Unlearning'
if not os.path.isdir(REPO_DIR):
    sh(f'git clone https://github.com/tiensinh2/CMF_Unlearning.git {REPO_DIR}')
else:
    sh(f'git -C {REPO_DIR} remote set-url origin https://github.com/tiensinh2/CMF_Unlearning.git')
    sh(f'git -C {REPO_DIR} pull origin main')
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
result = subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'],
                        capture_output=True, text=True)
REPO_COMMIT = result.stdout.strip() or 'main'
print('Repo commit:', REPO_COMMIT)

In [ ]:
# ── Must match exactly what NB1 used ─────────────────────────────────────────
DATASET        = 'cifar100'
ARCH           = 'resnet18'
NUM_CLASSES    = 100
SEEDS          = [0, 1, 2]
FORGET_CLASSES = [0, 1, 2, 3, 5]
TEST_MODE      = False

BATCH_SIZE     = 128
EPOCHS         = 300
PATIENCE       = 50
LR_INIT        = 1e-2
WEIGHT_DECAY   = 5e-4
MOMENTUM       = 0.9
WARMUP_EPOCHS  = 5
MIN_LR         = 1e-5

LP_MAX_EPOCHS  = 200 if DATASET.lower() == 'cifar100' else 50
LP_LR          = 1e-2
LP_BATCH_SIZE  = 256
NCC_BATCH_SIZE = 256

# Input dataset (flat layout — files at root level)
NB1_DATASET_DIR = '/kaggle/input/datasets/kiethe/cmf-nb1-cifar100'

# Output root — NB4a looks for:
#   {root}/cmf_benchmark_config.json
#   {root}/pre_train/theta_o_seed{s}.pt
#   {root}/splits/forget_indices_class{c}.json
CKPT_ROOT = '/kaggle/working/checkpoints/cmf_benchmark'
os.makedirs(f'{CKPT_ROOT}/pre_train', exist_ok=True)
os.makedirs(f'{CKPT_ROOT}/splits',    exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'DATASET={DATASET}  NUM_CLASSES={NUM_CLASSES}  FORGET_CLASSES={FORGET_CLASSES}')
print(f'NB1_DATASET_DIR exists: {os.path.isdir(NB1_DATASET_DIR)}')
print(f'device={device}')

In [ ]:
import torchvision, torchvision.transforms as transforms
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)),
])
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)),
])
full_train = torchvision.datasets.CIFAR100('/kaggle/working/data', train=True,
                                            download=True, transform=transform_train)
test_set   = torchvision.datasets.CIFAR100('/kaggle/working/data', train=False,
                                            download=True, transform=transform_test)
print(f'Train: {len(full_train)}  Test: {len(test_set)}')

In [ ]:
from models.resnet import ResNet18

def build_model():
    m = ResNet18(num_classes=NUM_CLASSES, dataset=DATASET)
    return m.to(device)

theta_o_models = {}
for seed in SEEDS:
    tag = f'theta_o_seed{seed}'
    ckpt_path = f'{NB1_DATASET_DIR}/{tag}.pt'
    assert os.path.exists(ckpt_path), f'Checkpoint not found: {ckpt_path}'
    ck = torch.load(ckpt_path, map_location=device)
    model = build_model()
    model.load_state_dict(ck['model_state_dict'])
    model.eval()
    theta_o_models[seed] = model
    print(f'  Loaded seed={seed} from {ckpt_path}')

print('All theta_o models loaded.')

In [ ]:
split_info = {}
for forget_class in FORGET_CLASSES:
    tag   = f'class{forget_class}'
    fpath = f'{NB1_DATASET_DIR}/forget_indices_{tag}.json'
    rpath = f'{NB1_DATASET_DIR}/retain_indices_{tag}.json'
    assert os.path.exists(fpath), f'Split not found: {fpath}'
    with open(fpath) as f: forget_idx = json.load(f)
    with open(rpath) as f: retain_idx = json.load(f)
    split_info[forget_class] = {'forget': forget_idx, 'retain': retain_idx}
    print(f'  class {forget_class}: forget={len(forget_idx)}  retain={len(retain_idx)}')

print('All splits loaded.')

In [ ]:
# ── Copy flat files into the subdirectory structure NB4a expects ─────────────
import shutil

# Copy theta_o checkpoints → pre_train/
for seed in SEEDS:
    src = f'{NB1_DATASET_DIR}/theta_o_seed{seed}.pt'
    dst = f'{CKPT_ROOT}/pre_train/theta_o_seed{seed}.pt'
    if not os.path.exists(dst):
        shutil.copy2(src, dst)
        print(f'  Copied theta_o_seed{seed}.pt → pre_train/')
    else:
        print(f'  pre_train/theta_o_seed{seed}.pt already exists')

# Copy split files → splits/
for forget_class in FORGET_CLASSES:
    tag = f'class{forget_class}'
    for prefix in ['forget_indices', 'retain_indices']:
        src = f'{NB1_DATASET_DIR}/{prefix}_{tag}.json'
        dst = f'{CKPT_ROOT}/splits/{prefix}_{tag}.json'
        if not os.path.exists(dst):
            shutil.copy2(src, dst)
            print(f'  Copied {prefix}_{tag}.json → splits/')
        else:
            print(f'  splits/{prefix}_{tag}.json already exists')

print('\nDirectory structure ready:')
for root, dirs, files in os.walk(CKPT_ROOT):
    level = root.replace(CKPT_ROOT, '').count(os.sep)
    print(f'  {"  "*level}{os.path.basename(root)}/')
    for f in files:
        print(f'  {"  "*(level+1)}{f}')


In [ ]:
# ── Section E: Evaluate Θ_o — Output / Linear Probe / NCC ───────────────────

@torch.no_grad()
def output_retain_forget(model, test_set, forget_indices):
    train_targets  = np.array(full_train.targets)
    forget_classes = sorted(set(train_targets[forget_indices].tolist()))
    model.eval()
    all_pred, all_true = [], []
    loader = torch.utils.data.DataLoader(
        test_set, batch_size=256, shuffle=False, num_workers=2)
    for x, y in loader:
        all_pred.extend(model(x.to(device)).argmax(1).cpu().tolist())
        all_true.extend(y.tolist())
    pred   = np.array(all_pred); true = np.array(all_true)
    f_mask = np.isin(true, forget_classes); r_mask = ~f_mask
    ret_acc = 100.0 * (pred[r_mask] == true[r_mask]).mean() if r_mask.any() else float('nan')
    for_acc = 100.0 * (pred[f_mask] == true[f_mask]).mean() if f_mask.any() else float('nan')
    return ret_acc, for_acc, forget_classes

@torch.no_grad()
def extract_features(model, loader):
    feats, labels = [], []
    buf = []
    def _hook(_m, _inp, out):
        buf.append(out.view(out.size(0), -1).detach().cpu())
    handle = model.avgpool.register_forward_hook(_hook)
    model.eval()
    for x, y in loader:
        buf.clear()
        _ = model(x.to(device))
        feats.append(buf[0]); labels.append(y)
    handle.remove()
    return torch.cat(feats, 0).float(), torch.cat(labels, 0).long()

def linear_probe_eval(model, train_loader_full, test_set, forget_classes):
    probe_model = copy.deepcopy(model).to(device)
    probe_model.eval()
    for p in probe_model.parameters(): p.requires_grad_(False)
    Xtr, ytr = extract_features(probe_model, train_loader_full)
    Xte, yte = extract_features(probe_model, torch.utils.data.DataLoader(
        test_set, batch_size=LP_BATCH_SIZE, shuffle=False, num_workers=2))
    clf = nn.Linear(Xtr.size(1), NUM_CLASSES).to(device)
    opt = optim.SGD(clf.parameters(), lr=LP_LR, momentum=0.9, weight_decay=0.0)
    dl  = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(Xtr, ytr), batch_size=LP_BATCH_SIZE, shuffle=True)
    torch.manual_seed(42)
    for _ in range(LP_MAX_EPOCHS):
        clf.train()
        for bx, by in dl:
            opt.zero_grad()
            nn.CrossEntropyLoss()(clf(bx.to(device)), by.to(device)).backward()
            opt.step()
    clf.eval()
    with torch.no_grad():
        pred_te = clf(Xte.to(device)).argmax(1).cpu().numpy()
    true_te = yte.numpy()
    f_mask = np.isin(true_te, forget_classes); r_mask = ~f_mask
    return (100.0 * (pred_te[r_mask] == true_te[r_mask]).mean() if r_mask.any() else float('nan'),
            100.0 * (pred_te[f_mask] == true_te[f_mask]).mean() if f_mask.any() else float('nan'))

def ncc_eval(model, train_loader_full, test_set, forget_classes):
    probe_model = copy.deepcopy(model).to(device)
    probe_model.eval()
    for p in probe_model.parameters(): p.requires_grad_(False)
    Xtr, ytr = extract_features(probe_model, train_loader_full)
    means = []
    for c in range(NUM_CLASSES):
        mask = (ytr == c)
        means.append(Xtr[mask].mean(0) if mask.any() else torch.zeros(Xtr.size(1)))
    M = torch.stack(means).to(device)
    Xte, yte = extract_features(probe_model, torch.utils.data.DataLoader(
        test_set, batch_size=NCC_BATCH_SIZE, shuffle=False, num_workers=2))
    Xte = Xte.to(device)
    pred  = torch.cdist(Xte.unsqueeze(0), M.unsqueeze(0)).squeeze(0).argmin(1).cpu().numpy()
    true  = yte.numpy()
    f_mask = np.isin(true, forget_classes); r_mask = ~f_mask
    return (100.0 * (pred[r_mask] == true[r_mask]).mean() if r_mask.any() else float('nan'),
            100.0 * (pred[f_mask] == true[f_mask]).mean() if f_mask.any() else float('nan'))

full_train_loader = torch.utils.data.DataLoader(
    full_train, batch_size=128, shuffle=False, num_workers=2)

eval_rows = []
for seed in SEEDS:
    model = theta_o_models[seed]
    for forget_class in FORGET_CLASSES:
        split          = split_info[forget_class]
        forget_idx     = split['forget']
        forget_classes = [forget_class]
        print(f'\n[seed={seed} forget_class={forget_class}]')
        out_ret, out_for, _ = output_retain_forget(model, test_set, forget_idx)
        print(f'  Output  retain={out_ret:.2f}%  forget={out_for:.2f}%')
        lp_ret, lp_for = linear_probe_eval(model, full_train_loader, test_set, forget_classes)
        print(f'  LP      retain={lp_ret:.2f}%  forget={lp_for:.2f}%')
        ncc_ret, ncc_for = ncc_eval(model, full_train_loader, test_set, forget_classes)
        print(f'  NCC     retain={ncc_ret:.2f}%  forget={ncc_for:.2f}%')
        eval_rows.append({
            'seed': seed, 'forget_class': forget_class,
            'out_retain': round(out_ret, 2), 'out_forget': round(out_for, 2),
            'lp_retain':  round(lp_ret,  2), 'lp_forget':  round(lp_for,  2),
            'ncc_retain': round(ncc_ret, 2), 'ncc_forget': round(ncc_for, 2),
        })

df_eval = pd.DataFrame(eval_rows)
print('\n=== Theta_o Evaluation ===')
print(df_eval.to_string(index=False))

eval_path = f'{CKPT_ROOT}/theta_o_eval.json'
with open(eval_path, 'w') as f:
    json.dump(eval_rows, f, indent=2)
print(f'\nSaved: {eval_path}')

In [ ]:
# ── Section F: Export cmf_benchmark_config.json ──────────────────────────────
config = {
    'repo_url':    'https://github.com/tiensinh2/CMF_Unlearning',
    'repo_commit': REPO_COMMIT,
    'dataset':     DATASET,
    'arch':        ARCH,
    'num_classes': NUM_CLASSES,
    'seeds':          SEEDS,
    'forget_classes': FORGET_CLASSES,
    'test_mode':      TEST_MODE,
    'ckpt_root':      CKPT_ROOT,
    'pretrain': {
        'batch_size':    BATCH_SIZE,
        'epochs':        EPOCHS,
        'patience':      PATIENCE,
        'lr_init':       LR_INIT,
        'weight_decay':  WEIGHT_DECAY,
        'momentum':      MOMENTUM,
        'warmup_epochs': WARMUP_EPOCHS,
        'min_lr':        MIN_LR,
        'lr_scheduler':  'cosine_with_warmup',
        'optimizer':     'SGD',
    },
    'split_protocol': 'whole_class',
    'split_files': {
        f'class{c}': {
            'forget': f'splits/forget_indices_class{c}.json',
            'retain': f'splits/retain_indices_class{c}.json',
        }
        for c in FORGET_CLASSES
    },
    'theta_o_ckpts': {str(s): f'pre_train/theta_o_seed{s}.pt' for s in SEEDS},
    'forget_baseline': 'oracle_forget_acc',
    'cmf_disclosed_deviation': (
        'recompute_cmf() L2-normalizes features before averaging. '
        'Paper Algorithm 1 uses raw z_theta(x). Intentional in our CMF variant.'
    ),
}

config_path = f'{CKPT_ROOT}/cmf_benchmark_config.json'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)
print('Saved:', config_path)

In [ ]:
# ── Section G: Summary ────────────────────────────────────────────────────────
print('=== NB1 Complete ===')
print(f'Checkpoint root : {CKPT_ROOT}')
print(f'Theta_o checkpoints : {[f"theta_o_seed{s}.pt" for s in SEEDS]}')
print(f'Split files     : forget_indices_class{{c}}.json  retain_indices_class{{c}}.json')
print(f'Config file     : cmf_benchmark_config.json')
print(f'Eval file       : theta_o_eval.json')
print()
print('=== Theta_o Evaluation --- Output / LP / NCC ===')
print(df_eval[['seed', 'forget_class',
               'out_retain', 'out_forget',
               'lp_retain',  'lp_forget',
               'ncc_retain', 'ncc_forget']].to_string(index=False))
print()
print('Next: Publish /kaggle/working/checkpoints/ as Kaggle dataset, then run NB4a.')